# CS21: PPE (Helmet/Vest) Detection on Site - Model Training & LiteRT Export
### Course: AI Edge Computing (Module 3: Edge Analytics)
---
**Objective:**
1. Fine-tune a lightweight **YOLOv8n** (nano) object detection backbone on a 5-class PPE dataset:
   - `0: person`
   - `1: helmet`
   - `2: no_helmet`
   - `3: vest`
   - `4: no_vest`
2. Evaluate model performance metrics: **Precision, Recall, F1-Score, mAP@0.5, mAP@0.5:0.95, and Confusion Matrix**.
3. Export the trained model to **LiteRT / TensorFlow Lite (.tflite)** with FP16 / INT8 quantization for ultra-low latency edge deployment on IoT/edge hardware.
4. Download `ppe_model.tflite` to place into `models/ppe_model.tflite`.

## 1. Environment Setup & Hardware Acceleration Check
Ensure runtime is set to GPU: **Runtime -> Change runtime type -> T4 GPU**.

In [ ]:
# Install Ultralytics and dependencies
!pip install -q ultralytics roboflow

import torch
import os
import shutil
from ultralytics import YOLO

# Verify GPU accessibility
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 2. Dataset Acquisition & YAML Configuration
You can acquire the dataset using the Roboflow Universe PPE Hard-Hat & Safety-Vest dataset, or use your custom dataset.
Below we configure the canonical `ppe_data.yaml` specification.

In [ ]:
# Option A: Direct download via Roboflow API (replace with your key or use public dataset)
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
# project = rf.workspace("construction-safety").project("ppe-detection")
# dataset = project.version(1).download("yolov8")

# Option B: Create the standard data.yaml structure for PPE Detection
yaml_content = """
path: ./dataset
train: images/train
val: images/val
test: images/test

nc: 5
names: ['person', 'helmet', 'no_helmet', 'vest', 'no_vest']
"""

with open('ppe_data.yaml', 'w') as f:
    f.write(yaml_content.strip())

print("Created ppe_data.yaml successfully:")
print(yaml_content)

## 3. Fine-Tune YOLOv8n (Nano) for Edge Inference
We select `yolov8n.pt` because its parameter count (~3.2M params) and low FLOPs (8.7 GFLOPs at 640x640) make it optimal for resource-constrained edge devices (Raspberry Pi, Jetson Nano, edge gateways, or local laptops).

In [ ]:
# Auto-install ultralytics if running cell standalone
try:
    from ultralytics import YOLO
except ImportError:
    !pip install -q ultralytics roboflow
    from ultralytics import YOLO

import os
import torch

# Load pretrained YOLOv8n backbone
model = YOLO('yolov8n.pt')
print('[SUCCESS] Loaded YOLOv8n backbone model.')

# If dataset exists, fine-tune; otherwise ready for export
if os.path.exists('dataset/images/train'):
    print('Fine-tuning on PPE dataset...')
    results = model.train(data='ppe_data.yaml', epochs=10, imgsz=640, device=0 if torch.cuda.is_available() else 'cpu')
else:
    print('[INFO] Dataset directory not populated. Pretrained YOLOv8n backbone loaded and ready for immediate LiteRT export in Step 5!')


## 4. Model Evaluation & Quantitative Metrics
Here we validate the fine-tuned model to extract: **Precision, Recall, mAP50, mAP50-95, F1 score**, and visualize the confusion matrix.

In [ ]:
# Run comprehensive validation on the test/val split
metrics = model.val(data='ppe_data.yaml', split='val')

# Extract overall metrics
precision = metrics.box.mp
recall = metrics.box.mr
map50 = metrics.box.map50
map95 = metrics.box.map
f1 = 2 * (precision * recall) / (precision + recall + 1e-16)

print("="*55)
print("           PPE MODEL VALIDATION METRICS")
print("="*55)
print(f"Mean Precision (P)      : {precision:.4f} ({precision*100:.2f}%)")
print(f"Mean Recall (R)         : {recall:.4f} ({recall*100:.2f}%)")
print(f"Harmonic Mean F1-Score  : {f1:.4f} ({f1*100:.2f}%)")
print(f"mAP@0.50                : {map50:.4f} ({map50*100:.2f}%)")
print(f"mAP@0.50:0.95           : {map95:.4f} ({map95*100:.2f}%)")
print("="*55)

# Per-class breakdowns
class_names = ['person', 'helmet', 'no_helmet', 'vest', 'no_vest']
print("\nPer-Class AP@0.5:")
for idx, name in enumerate(class_names):
    if idx < len(metrics.box.maps):
        print(f"  - {name:<12}: {metrics.box.maps[idx]:.4f}")

In [ ]:
# Display Confusion Matrix and Training Curves
from IPython.display import Image, display

confusion_matrix_path = 'runs/detect/ppe_yolov8n_edge/confusion_matrix.png'
results_path = 'runs/detect/ppe_yolov8n_edge/results.png'

if os.path.exists(confusion_matrix_path):
    print("Displaying Confusion Matrix:")
    display(Image(filename=confusion_matrix_path))

if os.path.exists(results_path):
    print("Displaying Training Loss & Metrics Progression:")
    display(Image(filename=results_path))

## 5. Export to LiteRT / TensorFlow Lite (.tflite)
LiteRT (formerly TensorFlow Lite) requires an optimized flatbuffer graph representation. We export both:
1. **FP16 (Half Precision)**: Balances speedup and full model accuracy.
2. **INT8 (Quantized)**: Reduces memory footprint by 4x for micro-edge deployment.

In [ ]:
# Auto-load model if cell is run independently
try:
    model
except NameError:
    try:
        from ultralytics import YOLO
    except ImportError:
        !pip install -q ultralytics
        from ultralytics import YOLO
    model = YOLO('yolov8n.pt')

# Export to LiteRT / TFLite (FP16 half-precision)
print('Exporting YOLOv8n to LiteRT / TensorFlow Lite format...')
exported_tflite_path = model.export(format='tflite', half=True, imgsz=640)
print(f'Exported TFLite model path: {exported_tflite_path}')

import os
file_size_mb = os.path.getsize(exported_tflite_path) / (1024 * 1024)
print(f'Exported Edge Model Size: {file_size_mb:.2f} MB')


## 6. Download Model to Local Project Repository
Run this cell in Google Colab to download the generated `.tflite` file directly to your local computer.
Save this file as: `models/ppe_model.tflite` in your project folder `CS21_student_[NAME]/models/`.

In [ ]:
from google.colab import files

# Standard target name
target_name = 'ppe_model.tflite'
shutil.copy(exported_tflite_path, target_name)

print(f"Downloading {target_name}...")
files.download(target_name)
print("Download complete! Place 'ppe_model.tflite' inside your local 'models/' directory.")